In [2]:
#We make sure to import all the necessary libraries for our analysis
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [3]:
#We read the datasets into pandas dataframes
df_customer=pd.read_csv('data/olist_customers_dataset.csv')
df_location=pd.read_csv('data/olist_geolocation_dataset.csv')
df_order_items=pd.read_csv('data/olist_order_items_dataset.csv')
df_order_payments=pd.read_csv('data/olist_order_payments_dataset.csv')
df_order_reviews=pd.read_csv('data/olist_order_reviews_dataset.csv')
df_orders=pd.read_csv('data/olist_orders_dataset.csv')
df_products=pd.read_csv('data/olist_products_dataset.csv')
df_sellers=pd.read_csv('data/olist_sellers_dataset.csv')
df_category_translation=pd.read_csv('data/product_category_name_translation.csv')


In [4]:
"""
Questions to answer:
1. VOLUME AND QUANTITY 
Which are the top 10 cities by number of orders?
Which product category is the best-selling?
How many orders per month? Is there a growth trend?
What is the average number of items per order?
"""

'\nQuestions to answer:\n1. VOLUME AND QUANTITY \nWhich are the top 10 cities by number of orders?\nWhich product category is the best-selling?\nHow many orders per month? Is there a growth trend?\nWhat is the average number of items per order?\n'

In [5]:
#Which are the top 10 cities by number of orders?
#Joining orders and customers information
df_customers_orders=df_orders.merge(df_customer, how='left', on='customer_id')
top_10_cities=df_customers_orders['customer_city'].value_counts().head(10)
print(top_10_cities)

customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
Name: count, dtype: int64


In [6]:
#Which product category is the best-selling?
#Joining Orders DataFrame, OrderItems DataFrame, Products Info DataFrame
df_orders_items_info=df_orders.merge(df_order_items,how='left',on='order_id')\
                        .merge(df_products,how='left', on='product_id' )
best_product=df_orders_items_info['product_category_name'].value_counts().head(1)
print(best_product)

product_category_name
cama_mesa_banho    11115
Name: count, dtype: int64


In [7]:
#How many orders per month? Is there a growth trend?
#Converting string column into date time data type
df_orders['order_purchase_timestamp']=pd.to_datetime(df_orders['order_purchase_timestamp'])
#Add a column that includes only year-month by using to_period('M')
df_orders['year_month']=df_orders['order_purchase_timestamp'].dt.to_period('M')
#Group by period and calculate de size
orders_per_month = df_orders.groupby('year_month').size()
print(orders_per_month)


year_month
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, dtype: int64


In [8]:
#For this answer re use the same DataFrame created before.
items_per_order=df_order_items.groupby('order_id').size()
avg_items=items_per_order.mean()
print(avg_items)

1.1417306873695092


In [9]:
"""
2. MONEY AND SALES 
What is the average order value?
Which categories generate the most revenue?
What is the correlation between price and quantity sold?
How much does shipping affect the final price?
"""

'\n2. MONEY AND SALES \nWhat is the average order value?\nWhich categories generate the most revenue?\nWhat is the correlation between price and quantity sold?\nHow much does shipping affect the final price?\n'

In [10]:
#What is the average order value?
#Total avg price
avg_price = df_order_items.groupby('order_id', as_index=False).agg(total_order=('price','sum'))['total_order'].mean()
#Details of the analisis
order_totals = df_order_items.groupby('order_id')['price'].sum()
print(f'Mean: R${order_totals.mean():.2f}')
print(f'Median: R${order_totals.median():.2f}')
print(f'Range: R${order_totals.min():.2f} - R${order_totals.max():.2f}')



Mean: R$137.75
Median: R$86.90
Range: R$0.85 - R$13440.00


In [11]:
#Which categories generate the most revenue?
revenue_by_category = df_order_items.merge(df_products, on='product_id') \
    .groupby('product_category_name', as_index=False).agg(
        total_revenue=('price', 'sum'),
        quantity_sold=('order_item_id', 'count'),
        avg_price=('price', 'mean'),
        num_products=('product_id', 'nunique')
    ).sort_values('total_revenue', ascending=False)
print(revenue_by_category)


            product_category_name  total_revenue  quantity_sold   avg_price  \
11                   beleza_saude     1258681.34           9670  130.163531   
66             relogios_presentes     1205005.68           5991  201.135984   
13                cama_mesa_banho     1036988.68          11115   93.296327   
32                  esporte_lazer      988048.97           8641  114.344285   
44         informatica_acessorios      911954.32           7827  116.513903   
..                            ...            ...            ...         ...   
41                         flores        1110.04             33   33.637576   
15                casa_conforto_2         760.27             30   25.342333   
17              cds_dvds_musicais         730.00             14   52.142857   
37  fashion_roupa_infanto_juvenil         569.85              8   71.231250   
67             seguros_e_servicos         283.29              2  141.645000   

    num_products  
11          2444  
66          1

In [12]:
#What is the correlation between price and quantity sold?
#Calculate how many items were sold by product
quantity_by_product=df_order_items.groupby('product_id',as_index=False).size().reset_index(names='quantity_sold')
print(quantity_by_product)
#Obtain price per product
price_by_product = df_order_items[['product_id', 'price']].drop_duplicates()
print(price_by_product)
product_analysis = quantity_by_product.merge(price_by_product, on='product_id')
correlation = product_analysis['price'].corr(product_analysis['quantity_sold'])
print(correlation)
print(f"Correlación: {correlation:.2f}")

####pending


       quantity_sold                        product_id  size
0                  0  00066f42aeeb9f3007548bb9d3f33c38     1
1                  1  00088930e925c41fd95ebfe695fd2655     1
2                  2  0009406fd7479715e4bef61dd91f2462     1
3                  3  000b8f95fcb9e0096488278317764d19     2
4                  4  000d9be29b5207b54e86aa1b1ac54872     1
...              ...                               ...   ...
32946          32946  fff6177642830a9a94a0f2cba5e476d1     2
32947          32947  fff81cc3158d2725c0655ab9ba0f712c     1
32948          32948  fff9553ac224cec9d15d49f5a263411f     1
32949          32949  fffdb2d0ec8d6a61f0a0a0db3f25b441     5
32950          32950  fffe9eeff12fcbd74a2f2b007dde0c58     1

[32951 rows x 3 columns]
                              product_id   price
0       4244733e06e7ecb4970a6e2683c13e61   58.90
1       e5f2d52b802189ee658865ca93d83a8f  239.90
2       c777355d18b72b67abbeef9df44fd0fd  199.00
3       7634da152a4610f1595efa32f14722fc   12.

In [13]:
#How much does shipping affect the final price?
#Join orders and orders_items
total_orders=df_order_items['price'].sum()
total_freight_value=df_order_items['freight_value'].sum()
freight_percent=(total_freight_value/total_orders)*100
print(freight_percent)
price_per_order=df_order_items.groupby('order_id')['price'].sum().reset_index()
freight_per_order=df_order_items.groupby('order_id')['freight_value'].sum().reset_index()
total_per_orders=price_per_order.merge(freight_per_order,on='order_id')
print(total_per_orders)
total_per_orders['freight_percent']=(total_per_orders['freight_value']/total_per_orders['price'])*100
print(total_per_orders['freight_percent'].mean())
print(total_per_orders['freight_percent'].median())
print(total_per_orders['freight_percent'].std())


16.56833852994542
                               order_id   price  freight_value
0      00010242fe8c5a6d1ba2dd792cb16214   58.90          13.29
1      00018f77f2f0320c557190d7a144bdd3  239.90          19.93
2      000229ec398224ef6ca0657da4fc703e  199.00          17.87
3      00024acbcdf0a6daa1e931b038114c75   12.99          12.79
4      00042b26cf59d7ce69dfabb4e55b4fd9  199.90          18.14
...                                 ...     ...            ...
98661  fffc94f6ce00a00581880bf54a75a037  299.99          43.41
98662  fffcd46ef2263f404302a634eb57f7eb  350.00          36.53
98663  fffce4705a9662cd70adb13d4a31832d   99.90          16.95
98664  fffe18544ffabc95dfada21779c9644f   55.99           8.72
98665  fffe41c64501cc87c801fd61db3f6244   43.00          12.79

[98666 rows x 3 columns]
30.83886871973845
22.437395659432386
31.47621932533826


In [14]:

"""3. CUSTOMER SATISFACTION ⭐
What is the average review rating? Is it improving?
Which categories have the best/worst ratings?
Is there a relationship between product price and rating?
Is there a relationship between delivery time and satisfaction?"""

'3. CUSTOMER SATISFACTION ⭐\nWhat is the average review rating? Is it improving?\nWhich categories have the best/worst ratings?\nIs there a relationship between product price and rating?\nIs there a relationship between delivery time and satisfaction?'

In [15]:
#What is the average review rating? Is it improving?
avg_rating=df_order_reviews['review_score'].mean().round(2)
print(avg_rating)
df_order_reviews['review_answer_timestamp']=pd.to_datetime(df_order_reviews['review_answer_timestamp'])
df_order_reviews['month_year']=df_order_reviews['review_answer_timestamp'].dt.to_period('M')
avg_rating_by_time=df_order_reviews.groupby('month_year')['review_score'].mean().reset_index()
print(avg_rating_by_time)

4.09
   month_year  review_score
0     2016-10      4.158940
1     2016-11      3.218182
2     2016-12      2.360000
3     2017-01      4.398990
4     2017-02      4.290936
5     2017-03      4.028743
6     2017-04      4.049929
7     2017-05      4.085424
8     2017-06      4.121590
9     2017-07      4.177275
10    2017-08      4.224007
11    2017-09      4.195181
12    2017-10      4.183840
13    2017-11      4.110751
14    2017-12      3.947011
15    2018-01      4.042245
16    2018-02      4.013451
17    2018-03      3.768025
18    2018-04      3.883581
19    2018-05      4.196196
20    2018-06      4.175935
21    2018-07      4.287429
22    2018-08      4.209052
23    2018-09      4.308544
24    2018-10      3.944444


In [ ]:
#Which categories have the best/worst ratings?
df_reviews_and_categories=df_order_reviews.merge(df_order_items,on='order_id',how='left')\
                            .merge(df_products,on='product_id',how='left')
df_categories_reviews=df_reviews_and_categories.groupby('product_category_name')['review_score'].mean().reset_index().sort_values(by='review_score',ascending=False)
print(df_categories_reviews.iloc[[0,-1]])



                            product_category_name  review_score
17                              cds_dvds_musicais      4.642857
37                  fashion_roupa_infanto_juvenil      4.500000
48                         livros_interesse_geral      4.446266
22             construcao_ferramentas_ferramentas      4.444444
41                                         flores      4.419355
..                                            ...           ...
55                              moveis_escritorio      3.493183
60                                       pc_gamer      3.333333
65  portateis_cozinha_e_preparadores_de_alimentos      3.266667
42                                fraldas_higiene      3.256410
67                             seguros_e_servicos      2.500000

[73 rows x 2 columns]


In [32]:
'''4. PATRONES TEMPORALES 📅
¿Cuál es el mes con más ventas? ¿Por qué?
¿Hay estacionalidad? (ej: Navidad, Black Friday)
¿Las órdenes están creciendo o cayendo?
¿El rating mejora o empeora con el tiempo?'''

'4. PATRONES TEMPORALES 📅\n¿Cuál es el mes con más ventas? ¿Por qué?\n¿Hay estacionalidad? (ej: Navidad, Black Friday)\n¿Las órdenes están creciendo o cayendo?\n¿El rating mejora o empeora con el tiempo?'

In [33]:
'''5. GEOGRAFÍA 🗺️
¿Cuáles son las 5 estados con más ventas?
¿Hay diferencia en precio/satisfacción por región?
¿Cuál es el costo de flete por región?'''

'5. GEOGRAFÍA 🗺️\n¿Cuáles son las 5 estados con más ventas?\n¿Hay diferencia en precio/satisfacción por región?\n¿Cuál es el costo de flete por región?'

In [34]:
'''6. COMPARACIONES 🔄
Productos caros vs baratos: ¿cuál se vende más?
¿Los productos más caros tienen mejor/peor rating?
¿Categoría A vende más que categoría B?'''

'6. COMPARACIONES 🔄\nProductos caros vs baratos: ¿cuál se vende más?\n¿Los productos más caros tienen mejor/peor rating?\n¿Categoría A vende más que categoría B?'